![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 07: Geospatial & Spatial Epidemiology
***
**Learning objectives**
- Implement circular spatial scan statistic (Kulldorff, 1997)
- Detect significant disease clusters with Monte Carlo permutation inference
- Apply elliptical scan for anisotropic clusters
- Evaluate cluster detection performance (sensitivity, specificity, PPV)
- Visualise detected clusters with uncertainty bounds
- Compare spatial scan vs LISA for cluster detection

**Estimated time:** 3 hours | **Level:** Advanced  
**Libraries:** `numpy`, `scipy`, `matplotlib`


## 1. Setup — With Planted Clusters

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from scipy.ndimage import gaussian_filter; from scipy.spatial import cKDTree
from scipy import stats; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod07", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)
NROW,NCOL=10,20; N=NROW*NCOL
row_idx=np.repeat(np.arange(NROW),NCOL); col_idx=np.tile(np.arange(NCOL),NROW)
lon=-120+col_idx*2.5+np.random.normal(0,0.3,N)
lat=28+row_idx*2.0+np.random.normal(0,0.2,N)
pct_poverty=10+8*np.random.beta(2,5,N)+3*np.sin(lon/20)
pm25=8+4*np.random.gamma(2,1,N); population=np.random.lognormal(10.5,1.0,N).astype(int)
def sp(vals,sigma=2):
    grid=np.zeros((NROW,NCOL))
    for r,c,v in zip(row_idx,col_idx,vals): grid[r,c]=v
    return gaussian_filter(grid,sigma=sigma)[row_idx,col_idx]
spatial_risk=sp(np.random.normal(0,1,N))
cvd_rate=(180+1.2*pct_poverty+0.5*pm25+15*spatial_risk+np.random.normal(0,12,N))
# Plant two known clusters
cluster1_mask = ((lon > -105) & (lon < -95) & (lat > 30) & (lat < 36))
cluster2_mask = ((lon > -85) & (lon < -75) & (lat > 38) & (lat < 44))
cvd_rate[cluster1_mask] += 35   # true hot spot 1
cvd_rate[cluster2_mask] += 25   # true hot spot 2
expected=(cvd_rate/100000)*population*5
observed=np.random.poisson(expected).astype(int)
smr=observed/expected.clip(1)
df=pd.DataFrame({"county_id":[f"C{i:03d}" for i in range(N)],
    "lon":lon,"lat":lat,"row":row_idx,"col":col_idx,
    "pct_poverty":pct_poverty,"pm25":pm25,"population":population,
    "cvd_rate":cvd_rate,"observed":observed,"expected":expected,"smr":smr,
    "true_cluster1":cluster1_mask,"true_cluster2":cluster2_mask})
print(f"N={N} | Cluster1: {cluster1_mask.sum()} counties | Cluster2: {cluster2_mask.sum()} counties")

## 2. Kulldorff Spatial Scan Statistic

The **spatial scan statistic** tests whether the number of cases within a circular window Z is higher than expected:

```
LLR(Z) = log[ (O_Z/E_Z)^O_Z * ((O-O_Z)/(E-E_Z))^(O-O_Z) ]
```

Where:
- O_Z, E_Z = observed and expected cases inside window Z
- O, E = total observed and expected
- The window that maximises LLR is the **most likely cluster**
- Significance is assessed via Monte Carlo permutation (randomise cases under H₀)

This is equivalent to the Poisson spatial scan statistic implemented in SaTScan software.


In [ ]:
from scipy.spatial import cKDTree

def scan_statistic_poisson(df, max_pop_fraction=0.5, n_perm=199, seed=42):
    """ Kulldorff circular spatial scan statistic (Poisson model). Tests circles of varying radii centred at each county. """
    coords   = np.column_stack([df.lon, df.lat])
    O_total  = df.observed.sum()
    E_total  = df.expected.sum()
    n        = len(df)
    max_pop  = df.population.sum() * max_pop_fraction

    # Precompute all pairwise distances
    tree = cKDTree(coords)
    dists_all, _ = tree.query(coords, k=n)  # all neighbours sorted by distance

    def llr_poisson(O_z, E_z):
        if O_z == 0 or E_z == 0 or O_z <= E_z:
            return 0
        O_out = O_total - O_z
        E_out = E_total - E_z
        if O_out <= 0 or E_out <= 0 or O_out <= E_out:
            return 0
        llr = O_z * np.log(O_z/E_z) + O_out * np.log(O_out/E_out)
        return max(0, llr)

    # Scan all candidate windows
    best_llr = 0; best_center = 0; best_radius = 0; best_inside = []
    all_windows = []

    for center_i in range(n):
        sorted_dists, sorted_idxs = tree.query(coords[center_i], k=n)
        O_z = 0; E_z = 0; pop_z = 0
        for k in range(1, n):
            j = sorted_idxs[k]
            pop_z += df.population.iloc[j]
            if pop_z > max_pop:
                break
            O_z += df.observed.iloc[j]
            E_z += df.expected.iloc[j]
            llr = llr_poisson(O_z, E_z)
            if llr > best_llr:
                best_llr = llr
                best_center = center_i
                best_radius = sorted_dists[k]
                best_inside = sorted_idxs[1:k+1].tolist()
            all_windows.append({"center":center_i,"radius":sorted_dists[k],"llr":llr})

    # Monte Carlo permutation test
    rng = np.random.default_rng(seed)
    mc_llrs = np.zeros(n_perm)
    O_arr = df.observed.values; E_arr = df.expected.values
    for b in range(n_perm):
        O_perm = rng.multinomial(O_total, E_arr/E_total)
        best_b = 0
        for ci in range(n):
            _, sorted_j = tree.query(coords[ci], k=n)
            Oz=0; Ez=0; popz=0
            for k in range(1,n):
                j=sorted_j[k]; popz+=df.population.iloc[j]
                if popz>max_pop: break
                Oz+=O_perm[j]; Ez+=E_arr[j]
                l=llr_poisson(Oz,Ez)
                if l>best_b: best_b=l
        mc_llrs[b] = best_b

    p_value = (mc_llrs >= best_llr).mean()
    return {"best_llr":best_llr, "best_center":best_center,
            "best_radius":best_radius, "best_inside":best_inside,
            "p_value":p_value, "mc_llrs":mc_llrs}

print("Running spatial scan statistic (may take 1-2 minutes)...")
scan_res = scan_statistic_poisson(df, max_pop_fraction=0.40, n_perm=199)
print(f"Primary cluster: centre={df.iloc[scan_res['best_center']]['county_id']} | "
      f"radius={scan_res['best_radius']:.2f}° | "
      f"n={len(scan_res['best_inside'])} counties | "
      f"LLR={scan_res['best_llr']:.2f} | p={scan_res['p_value']:.4f}")


In [ ]:
# Cluster map
inside_idx = set(scan_res["best_inside"])
df["in_cluster1"] = [i in inside_idx for i in range(len(df))]

# Compute SMR inside vs outside cluster
smr_in  = df.loc[df.in_cluster1, "smr"].mean()
smr_out = df.loc[~df.in_cluster1, "smr"].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# True clusters vs detected
true_any = df.true_cluster1 | df.true_cluster2
colors_det = ["#D65F5F" if v else "#E0E0E0" for v in df.in_cluster1]
colors_true = ["#FF6B6B" if t1 else "#4878CF" if t2 else "#E0E0E0"
               for t1,t2 in zip(df.true_cluster1, df.true_cluster2)]

axes[0].scatter(df.lon, df.lat, c=colors_true, s=150, edgecolors="white", linewidth=0.2)
axes[0].set_title("TRUE clusters (planted) Red=Cluster1 (+35/100k) | Blue=Cluster2 (+25/100k)")

axes[1].scatter(df.lon, df.lat, c=colors_det, s=150, edgecolors="white", linewidth=0.2)
# Draw detected cluster circle
cx = df.iloc[scan_res["best_center"]].lon
cy = df.iloc[scan_res["best_center"]].lat
circle = plt.Circle((cx,cy), scan_res["best_radius"], fill=False,
                     edgecolor="black", lw=2.5, linestyle="--")
axes[1].add_patch(circle)
axes[1].set_title(f"DETECTED cluster (scan statistic) LLR={scan_res['best_llr']:.2f}, p={scan_res['p_value']:.3f} "
                  f"SMR inside={smr_in:.3f} vs outside={smr_out:.3f}")

for ax in axes:
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_xlim(df.lon.min()-2, df.lon.max()+2)
    ax.set_ylim(df.lat.min()-1, df.lat.max()+1)
plt.tight_layout(); plt.savefig("/tmp/mod07/scan_cluster_map.png", bbox_inches="tight"); plt.show()


## 3. Cluster Detection Performance

In [ ]:
# Evaluate detection accuracy
detected = np.array(df.in_cluster1, dtype=bool)
true_any = (df.true_cluster1 | df.true_cluster2).values.astype(bool)

tp = (detected & true_any).sum()
fp = (detected & ~true_any).sum()
fn = (~detected & true_any).sum()
tn = (~detected & ~true_any).sum()

sensitivity = tp/(tp+fn) if tp+fn>0 else 0
specificity  = tn/(tn+fp) if tn+fp>0 else 0
ppv          = tp/(tp+fp) if tp+fp>0 else 0
npv          = tn/(tn+fn) if tn+fn>0 else 0

print("Cluster Detection Performance:")
print(f"  True positives  : {tp}")
print(f"  False positives : {fp}")
print(f"  False negatives : {fn} (true cluster counties missed)")
print(f"  True negatives  : {tn}")
print(f"  Sensitivity     : {sensitivity:.3f}")
print(f"  Specificity     : {specificity:.3f}")
print(f"  PPV             : {ppv:.3f}")
print(f"  NPV             : {npv:.3f}")
print()

# SMR RR inside detected cluster
obs_in  = df.loc[detected, "observed"].sum()
exp_in  = df.loc[detected, "expected"].sum()
obs_out = df.loc[~detected, "observed"].sum()
exp_out = df.loc[~detected, "expected"].sum()
rr = (obs_in/exp_in) / (obs_out/exp_out)
print(f"Relative risk (inside vs outside detected cluster): {rr:.3f}")
print(f"Expected RR from planted effect: ~{1+35/180:.2f} to ~{1+25/180:.2f}")


## 4. Permutation Distribution & Inference

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monte Carlo distribution
axes[0].hist(scan_res["mc_llrs"], bins=30, color="#AEC6CF", edgecolor="white", alpha=0.9)
axes[0].axvline(scan_res["best_llr"], color="#D65F5F", lw=2.5,
                label=f"Observed LLR={scan_res['best_llr']:.2f}")
axes[0].set_xlabel("Spatial scan LLR"); axes[0].set_ylabel("Count")
axes[0].set_title(f"Monte Carlo Null Distribution (B=199) p={scan_res['p_value']:.4f}")
axes[0].legend(fontsize=9)

# Cluster size sensitivity
cluster_sizes = range(5, 55, 5)
llrs_by_size = []
for max_in in cluster_sizes:
    subset = df.head(max_in)
    O_z = subset.observed.sum(); E_z = subset.expected.sum()
    O_out = df.observed.sum()-O_z; E_out = df.expected.sum()-E_z
    if O_z>0 and E_z>0 and O_z>E_z and O_out>0 and E_out>0 and O_out>E_out:
        llr = O_z*np.log(O_z/E_z) + O_out*np.log(O_out/E_out)
    else:
        llr = 0
    llrs_by_size.append(max(0,llr))
axes[1].plot(cluster_sizes, llrs_by_size, "o-", color="#4878CF", lw=2)
axes[1].set_xlabel("Window size (# counties)"); axes[1].set_ylabel("LLR")
axes[1].set_title("LLR by Scanning Window Size")

plt.tight_layout(); plt.savefig("/tmp/mod07/scan_inference.png", bbox_inches="tight"); plt.show()


## 5. LISA vs Scan Statistic Comparison

In [ ]:
# Compare LISA HH cluster with scan statistic cluster
# LISA (from NB-02)
def local_morans_simple(y, W_bin, n_perm=199, seed=42):
    n=len(y); z=y-y.mean(); var_z=z.var()
    # Row-standardise
    rs=W_bin.sum(axis=1,keepdims=True); rs[rs==0]=1; W_rs=W_bin/rs
    Wz=W_rs@z; I_local=z*Wz/var_z
    rng=np.random.default_rng(seed)
    I_perm=np.array([(rng.permutation(z)*(W_rs@rng.permutation(z))/var_z) for _ in range(n_perm)])
    p_vals=(np.abs(I_perm)>=np.abs(I_local)).mean(axis=0)
    z_lag=Wz/z.std(); z_std=z/z.std()
    quad=np.where((z_std>0)&(z_lag>0),"HH",np.where((z_std<0)&(z_lag<0),"LL","Other"))
    cluster=np.where(p_vals<0.05,quad,"NS")
    return cluster

def build_queen_binary(row_idx,col_idx,nrow,ncol):
    N=len(row_idx); W=np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            if i!=j and max(abs(row_idx[i]-row_idx[j]),abs(col_idx[i]-col_idx[j]))<=1:
                W[i,j]=1
    return W

W_bin = build_queen_binary(df.row.values,df.col.values,NROW,NCOL)
lisa_clusters = local_morans_simple(df.cvd_rate.values, W_bin)
df["lisa_hh"] = (lisa_clusters=="HH")

# Detection performance comparison
fig, axes = plt.subplots(1,2,figsize=(14,5))
colors_lisa = ["#D65F5F" if v else "#E0E0E0" for v in df.lisa_hh]
colors_scan = ["#D65F5F" if v else "#E0E0E0" for v in df.in_cluster1]
for ax,colors,method in [(axes[0],colors_lisa,"LISA (HH, p<0.05)"),(axes[1],colors_scan,"Scan Statistic")]:
    ax.scatter(df.lon,df.lat,c=colors,s=150,edgecolors="white",linewidth=0.2)
    true_cov = (np.array(colors)!="#E0E0E0") & true_any
    n_det = (np.array(colors)!="#E0E0E0").sum()
    n_tp  = true_cov.sum()
    ax.set_title(f"{method} {n_det} flagged | {n_tp} true positives captured")
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
plt.suptitle("Cluster Detection: LISA vs Spatial Scan Statistic", fontsize=12)
plt.tight_layout(); plt.savefig("/tmp/mod07/lisa_vs_scan.png",bbox_inches="tight"); plt.show()


## 6. Knowledge Check
1. The scan statistic detects a cluster with p=0.03 (200 permutations). How precise is this p-value, and what would you report?
2. Your detected cluster has PPV=0.65. In a public health context, what does this mean for resource allocation decisions?
3. The maximum window size is set to 50% of population. Why is there an upper bound, and what is the trade-off?
4. LISA detects 18 HH counties but the scan statistic finds a different 15-county cluster. Why might they differ?
5. Implement a secondary cluster search: after removing the primary cluster, re-run the scan to find the next most likely cluster.


In [ ]:
# Exercise 5 — secondary cluster detection
# Remove primary cluster and search again
df_secondary = df.copy()
df_secondary = df_secondary[~df_secondary.in_cluster1].reset_index(drop=True)
print(f"Searching for secondary cluster in {len(df_secondary)} remaining counties...")
scan_sec = scan_statistic_poisson(df_secondary, max_pop_fraction=0.40, n_perm=99, seed=99)
sec_center_county = df_secondary.iloc[scan_sec["best_center"]]["county_id"]
sec_n = len(scan_sec["best_inside"])
print(f"Secondary cluster: centre={sec_center_county} | n={sec_n} counties | "
      f"LLR={scan_sec['best_llr']:.2f} | p={scan_sec['p_value']:.3f}")
if scan_sec["p_value"] < 0.05:
    print("Secondary cluster is statistically significant!")
    # Check overlap with true cluster 2
    inside_sec = set(scan_sec["best_inside"])
    true_cl2_idx = df_secondary[df_secondary.true_cluster2].index.tolist()
    overlap = len(set(true_cl2_idx) & inside_sec)
    print(f"Overlap with true cluster 2: {overlap}/{df_secondary.true_cluster2.sum()} counties")
else:
    print("No significant secondary cluster found.")


***
**Next → NB-06: Environmental Epidemiology & Spatial Exposure**
